In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.adapter.graph.knowledge_base import KnowledgeBase
from src.common.sementic_setting import settings as sementic_settings

kb = KnowledgeBase()

import re
import unicodedata
from typing import List, Dict, Any

def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("; ", ". ")
    text = text.replace(" - ", ". ")
    return text

def split_clauses(text: str) -> List[str]:
    text = normalize_text(text)
    conj_pattern = r"\b(?:%s)\b" % "|".join(map(re.escape, sorted(sementic_settings.CONJUNCTION, key=len, reverse=True)))
    parts = re.split(rf"[.!?;\n,]+|{conj_pattern}", text)
    clauses = [p.strip() for p in parts if p and p.strip()]
    return clauses

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

all_indicators = []
for virtue, indicators in kb.student_assessments.define["Phẩm chất chủ yếu"].items():
    for ind in indicators:
        all_indicators.append({
            "virtue": virtue,
            "indicator": ind
        })

indicator_texts = [x["indicator"] for x in all_indicators]
indicator_embs = model.encode(indicator_texts, convert_to_tensor=True)

def classify_comment(txt):
    clauses = split_clauses(txt)
    print(clauses)
    clause_embs = model.encode(clauses, convert_to_tensor=True)

    for i, clause in enumerate(clauses):
        sims = util.cos_sim(clause_embs[i], indicator_embs)[0]
        best_idx = sims.argmax().item()
        print(clause, all_indicators[best_idx], float(sims[best_idx]))
# Demo
txt = "Em luôn đoàn kết và yêu                     mến bạn bè. Em có tính trung thực và chấp hành tốt nội qui lớp học."
print(classify_comment(txt))

d:\Work\Iu\hoc-ba-so\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1463.26it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['em luôn đoàn kết', 'yêu mến bạn bè', 'em có tính trung thực', 'chấp hành tốt nội qui lớp học']
em luôn đoàn kết {'virtue': 'Chăm chỉ', 'indicator': ' Thường xuyên tham gia các công việc của gia đình vừa sức với bản thân.'} 0.4616358280181885
yêu mến bạn bè {'virtue': 'Nhân ái', 'indicator': 'Yêu quý bạn bè, thầy cô; quan tâm, động viên, khích lệ bạn bè.'} 0.7152847051620483
em có tính trung thực {'virtue': 'Trung thực', 'indicator': 'Trung thực, thật thà, ngay thẳng trong học tập, lao động và sinh hoạt hằng ngày; mạnh dạn nói lên ý kiến của mình.'} 0.546326756477356
chấp hành tốt nội qui lớp học {'virtue': 'Trách nhiệm', 'indicator': 'Nhắc nhở bạn bè chấp hành nội quy trường lớp; nhắc nhở người thân chấp hành các quy định, quy ước nơi công cộng.'} 0.7119568586349487
None
